# Watching a black hole form on a laptop
A verified Python lab on spherical scalar-field collapse (units $G=c=1$). Put `nr_lab.py` in the same folder.

**Rule of the lab:** nothing is interpreted physically until the self-checks pass.

| Stage | You do | Learning goal |
|---|---|---|
| 0 | run the self-checks | verification comes before discovery |
| 1 | watch a pulse disperse, then collapse | time dilation and compactness $2m/r$ |
| 2 | test Birkhoff's theorem | exterior of a spherical matter distribution |
| 3 | bisect for the critical amplitude | threshold of black-hole formation |
| 4 | break the code on purpose | what errors look like |

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time
import nr_lab as L

## Stage 0 - self-checks (about 25 s)
Three automatic checks: second-order convergence against an exact flat-space solution, Birkhoff's theorem, and the critical amplitude.

In [ ]:
ok, report = L.selfcheck()

## Stage 1 - disperse or collapse?
An ingoing Gaussian pulse of amplitude `p`. Compare central lapse $\alpha(t,0)$ (clock rate at the centre) and the largest compactness $2m/r$ (a horizon forms at 1).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for p in (0.012, 0.030):
    t_list, lapse, comp = [], [], []
    tt = 0.0
    for tmax in np.arange(0.5, 14.1, 0.5):
        o = L.evolve(L.gaussian(p), dr=0.04, tmax=tmax, stop_bh=False)
        t_list.append(o['t']); lapse.append(o['al'][0]); comp.append((2*o['m']/o['r']).max())
    ax[0].semilogy(t_list, lapse, label=f'p={p}'); ax[1].plot(t_list, comp, label=f'p={p}')
ax[0].set_xlabel('t'); ax[0].set_ylabel('central lapse'); ax[0].legend()
ax[1].axhline(1, color='k', lw=.6); ax[1].set_xlabel('t'); ax[1].set_ylabel('max 2m/r'); ax[1].legend(); plt.tight_layout(); plt.show()

**Question.** The lapse at the centre drops for both amplitudes. What distinguishes the two cases afterwards?

## Stage 2 - Birkhoff's theorem
Outside the matter, $m(r)$ should be constant and $\alpha^2 = 1-2M/r$.

In [ ]:
o = L.evolve(L.gaussian(0.012), R=30., dr=0.04, tmax=10., stop_bh=False)
S = o['Phi']**2 + o['Pi']**2; edge = o['r'][np.max(np.where(S > 1e-14)[0])] + 0.1
vac = (o['r'] > edge) & (o['r'] < 26); M = o['m'][vac][0]
print(f"M = {M:.6f}; spread of m(r) in vacuum = {(o['m'][vac].max()-o['m'][vac].min())/M:.1e}")
print(f"max |alpha^2 - (1-2M/r)| = {np.max(np.abs(o['al'][vac]**2-(1-2*M/o['r'][vac]))):.1e}")
plt.plot(o['r'], o['m']); plt.xlabel('r'); plt.ylabel('m(r)'); plt.show()

## Stage 3 - critical amplitude (about 20 s at coarse resolution)

In [ ]:
t0 = time.time(); lo, hi = L.bisect(L.gaussian, 0.015, 0.025, n=18, dr=0.04)
print(f"p* = {lo:.5f} (bracket width {hi-lo:.1e}), {time.time()-t0:.0f} s")

**Exercise.** Repeat with `dr=0.02` and `dr=0.01`. Does `p*` converge? Then record the black-hole mass `mbh` for `p = p*(1+delta)` and fit $M\propto\delta^{\gamma}$ (literature value about 0.37).

## Stage 4 - break it on purpose
(a) In `rhs`, change the origin parity `-1.` to `+1.`; (b) set `cfl=2.0`; (c) remove the absorbing layer (`sponge = 0*r`). Record what changes, and which self-check notices.

In [ ]:
o = L.evolve(L.gaussian(0.012), dr=0.04, tmax=20., cfl=2.0, stop_bh=False)
print('cfl = 2.0 -> max |Phi| =', np.abs(o['Phi']).max(), '(healthy run: about 6e-3)')